# Phase 4: CDC Diabetes Health Indicators - Data Understanding
**Project**: AI-Powered Health Risk Prediction and Monitoring System  
**Author**: Abhiram (AI/ML Engineer)  
**Dataset Source**: UCI Machine Learning Repository (Dataset #891) / CDC BRFSS 2015  
**Objective**: Ingest, validate schema, inspect data quality, analyze target distribution, and compute descriptive statistics for the CDC Diabetes dataset.

In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
from src.preprocessing.data_loader import load_cdc_diabetes_data
from src.features.cdc_diabetes_metadata import CDC_FEATURE_METADATA, BINARY_FEATURES, ORDINAL_FEATURES, CONTINUOUS_NUMERICAL_FEATURES

# Load primary binary dataset and 3-class dataset
df_binary = load_cdc_diabetes_data(variant='binary')
df_012 = load_cdc_diabetes_data(variant='012')
print('Binary Dataset Shape:', df_binary.shape)
print('Multiclass Dataset Shape:', df_012.shape)

## 1. Schema, Datatypes & Missing Values

In [ ]:
schema_df = pd.DataFrame([
    {
        'Column': col,
        'Dtype': str(df_binary[col].dtype),
        'Unique Values': df_binary[col].nunique(),
        'Missing Values': df_binary[col].isnull().sum(),
        'Role': CDC_FEATURE_METADATA.get(col, {}).get('role', 'Unknown'),
        'Feature Type': CDC_FEATURE_METADATA.get(col, {}).get('type', 'Unknown')
    }
    for col in df_binary.columns
])
schema_df

## 2. Target Variable Analysis: Diabetes_binary vs Diabetes_012
We examine class balance and the relationship between the binary and multiclass target encodings.

In [ ]:
target_summary = pd.DataFrame({
    'Count': df_binary['Diabetes_binary'].value_counts().sort_index(),
    'Percentage (%)': (df_binary['Diabetes_binary'].value_counts(normalize=True).sort_index() * 100).round(2)
})
print('=== Binary Target Distribution ===')
print(target_summary)

print('\n=== Cross-tabulation (Diabetes_012 vs Diabetes_binary) ===')
ct = pd.crosstab(df_012['Diabetes_012'], df_binary['Diabetes_binary'], margins=True)
print(ct)

## 3. Duplicate Rows & Data Quality Check

In [ ]:
features = [c for c in df_binary.columns if c != 'Diabetes_binary']
dup_exact = int(df_binary.duplicated().sum())
dup_features = int(df_binary.duplicated(subset=features).sum())
unique_profiles = int(len(df_binary.groupby(features)))
conflicts = int((df_binary.groupby(features)['Diabetes_binary'].nunique() > 1).sum())

print(f'Exact Duplicate Rows: {dup_exact:,} ({dup_exact/len(df_binary)*100:.2f}%)')
print(f'Feature Duplicate Rows: {dup_features:,} ({dup_features/len(df_binary)*100:.2f}%)')
print(f'Unique Respondent Profiles: {unique_profiles:,}')
print(f'Profiles with Discrepant Target Status: {conflicts:,} ({conflicts/unique_profiles*100:.2f}%)')

## 4. Descriptive Statistics for Continuous and Numerical Variables

In [ ]:
num_df = df_binary[CONTINUOUS_NUMERICAL_FEATURES].describe().T
num_df['skewness'] = df_binary[CONTINUOUS_NUMERICAL_FEATURES].skew()
num_df['kurtosis'] = df_binary[CONTINUOUS_NUMERICAL_FEATURES].kurtosis()
num_df